# BT4012 Kaggle Competiton (Final)

In [3]:
import os, time, warnings
import numpy as np
import pandas as pd
import scipy.sparse as sp
from scipy.sparse.csgraph import connected_components
from scipy.stats import rankdata
from sklearn.metrics import average_precision_score as ap, roc_auc_score as auc
import lightgbm as lgb
import torch, torch.nn as nn, torch.nn.functional as Fn
from torch_geometric.nn import GCNConv, SAGEConv, GATConv
from torch_geometric.utils import to_undirected, k_hop_subgraph

warnings.filterwarnings('ignore')
torch.set_num_threads(8)
SEED = 42
np.random.seed(SEED); torch.manual_seed(SEED)

DATA = r'bt4012/bt-4012-competition-2026'
t0 = time.time()


## 1. Load data

In [4]:
train = pd.read_csv(f'{DATA}/train.csv')
test = pd.read_csv(f'{DATA}/test.csv')
edges = pd.read_csv(f'{DATA}/txs_edgelist.csv')
sample = pd.read_csv(f'{DATA}/sample_submission.csv')
FEATS = [c for c in train.columns if c.startswith('feat_')]
lab = train[train.label.notna()].copy()
print(f'[{time.time()-t0:.0f}s] data loaded: train {train.shape} test {test.shape} labeled={len(lab)}')


[11s] data loaded: train (141772, 168) test (15329, 168) labeled=31235


## 2. Clock-feature drop ("old" features, cleaned)

A raw feature nearly perfectly correlated with `time_step` is dangerous for a model that must
extrapolate 14 steps past training -- computed live via Spearman correlation, same as the earlier
notebook, so the two stay comparable.

In [5]:
X_all = pd.concat([train[['time_step'] + FEATS], test[['time_step'] + FEATS]])
corr_t = X_all[FEATS + ['time_step']].corr('spearman')['time_step'].drop('time_step').abs().sort_values(ascending=False)
DRIFT = corr_t.head(6).index.tolist()
F_OLD = [f for f in FEATS if f not in DRIFT]
print(f'[{time.time()-t0:.0f}s] dropped {len(DRIFT)} clock feats: {DRIFT} | raw ("old") features kept: {len(F_OLD)}')


[69s] dropped 6 clock feats: ['feat_101', 'feat_103', 'feat_136', 'feat_100', 'feat_139', 'feat_137'] | raw ("old") features kept: 159


## 3. Feature Engineering

The graph is homophilic. Features are engineered to capture graph features and homophilic behaviour

In [6]:
ids = np.unique(np.concatenate([train.txId, test.txId, edges.txId1, edges.txId2]))
N = len(ids)
pos = pd.Series(np.arange(N), index=ids)
src = pos[edges.txId1].values; dst = pos[edges.txId2].values

A = sp.csr_matrix((np.ones(len(src)), (src, dst)), shape=(N, N))
AT = A.T.tocsr()
U = ((A + AT) > 0).astype(float).tocsr()
U2 = (U @ U).tocsr()

vis = np.zeros(N)
vis[pos[lab.txId].values] = 1
vis[pos[test.txId].values] = 1

G = pd.DataFrame({'txId': ids})
G['deg_out'] = np.asarray(A.sum(1)).ravel();  G['deg_in'] = np.asarray(AT.sum(1)).ravel()
G['vis_out'] = A @ vis;  G['vis_in'] = AT @ vis
G['unk_out'] = G.deg_out - G.vis_out;  G['unk_in'] = G.deg_in - G.vis_in
G['deg_und'] = np.asarray(U.sum(1)).ravel()
G['hop2'] = np.asarray((U @ U).sum(1)).ravel()
G['nb_deg_mean'] = (U @ G.deg_und.values) / np.maximum(G.deg_und, 1)
G['nb_vis_frac'] = (U @ vis) / np.maximum(G.deg_und, 1)

vb = vis.astype(bool); ev = vb[src] & vb[dst]
V = sp.csr_matrix((np.ones(ev.sum()), (src[ev], dst[ev])), shape=(N, N))
_, cl = connected_components(V, directed=False); cs = np.bincount(cl)
vin = np.asarray(V.sum(0)).ravel(); vout = np.asarray(V.sum(1)).ravel()
chain = ((vin == 1) & (vout == 1)).astype(float)
G['vcomp_size'] = np.where(vb, cs[cl], 0)
G['vcomp_chainfrac'] = np.where(vb, np.bincount(cl, chain)[cl] / cs[cl], 0)

_, cl2 = connected_components(A, directed=False); cs2 = np.bincount(cl2)
G['comp_visfrac'] = np.bincount(cl2, vis)[cl2] / cs2[cl2]

STRUCT_NEW = [c for c in G.columns if c != 'txId']
L = lab.merge(G, on='txId'); L['ix'] = pos[L.txId].values; L = L.reset_index(drop=True)
T = test.merge(G, on='txId'); T['ix'] = pos[T.txId].values
XF_ALL = F_OLD + STRUCT_NEW
print(f'[{time.time()-t0:.0f}s] {len(STRUCT_NEW)} graph ("new") features built | XF_ALL (old+new): {len(XF_ALL)} cols')


[70s] 13 graph ("new") features built | XF_ALL (old+new): 172 cols


## 4. Validation

In [7]:
def split(cut, hor):
    a = L[L.time_step <= cut].reset_index(drop=True)
    b = L[(L.time_step > cut) & (L.time_step <= cut + hor)].reset_index(drop=True)
    return a, b

WINDOWS = {
    'holdout_7@28':  (28, 7),
    'stress_7@22':   (22, 7),
    'horizon14@14':  (14, 14),
    'horizon14@17':  (17, 14),
    'horizon14@21':  (21, 14),
}
HARD_WINDOWS = ['stress_7@22', 'horizon14@14', 'horizon14@17', 'horizon14@21']

for nm, (cut, hor) in WINDOWS.items():
    a, b = split(cut, hor)
    print(f'{nm:16s} train<= {cut:2d}  score {cut+1:2d}-{cut+hor:2d}  n_train={len(a):6d} n_score={len(b):6d} illicit_rate={b.label.mean():.3f}')


holdout_7@28     train<= 28  score 29-35  n_train= 25207 n_score=  6028 illicit_rate=0.183
stress_7@22      train<= 22  score 23-29  n_train= 21293 n_score=  5088 illicit_rate=0.165
horizon14@14     train<= 14  score 15-28  n_train= 14896 n_score= 10311 illicit_rate=0.149
horizon14@17     train<= 17  score 18-31  n_train= 16855 n_score= 10760 illicit_rate=0.156
horizon14@21     train<= 21  score 22-35  n_train= 19530 n_score= 11705 illicit_rate=0.151


## 5. Shared LGBM + neighbor-prediction stacking machinery

Takes p, the model’s predicted fraud probability for every node (NaN if no prediction), and builds new features summarizing each node’s neighbors.

has marks which nodes have a prediction; pv is predictions with NaN replaced by 0.
AT, A, U, U2 are adjacency matrices: M[i, j] = 1 if j is a neighbor of i.
A: outgoing edges, AT: incoming edges, U: either direction, U2: neighbors-of-neighbors (2 hops).
For each matrix:
c = M @ has counts how many neighbors have predictions.
mean: the average predicted probability across those neighbors.
max: the highest neighbor’s probability, i.e. “is any neighbor very suspicious?”
If a node has no predicted neighbors, the value is −1.
Returns these features only for the rows in ix.

The idea: if your neighbors look like fraud, you’re more likely to be fraud.

In [8]:
BASE_LGB = dict(n_estimators=600, learning_rate=0.03, num_leaves=31, subsample=0.8,
                subsample_freq=1, colsample_bytree=0.5, verbose=-1, n_jobs=8)


def lgbm(Xd, y, w, seed, params=BASE_LGB):
    return lgb.LGBMClassifier(**{**params, 'random_state': seed}).fit(Xd, y, sample_weight=w)


def nb_feats(p, ix):
    has = ~np.isnan(p); pv = np.nan_to_num(p); out = {}
    for nm, M in [('in', AT), ('out', A), ('u', U)]:
        c = M @ has
        out[f'np_{nm}_mean'] = np.where(c > 0, (M @ pv) / np.maximum(c, 1), -1)
        out[f'np_{nm}_max'] = np.where(c > 0, M.multiply(pv[None, :]).max(1).toarray().ravel(), -1)
    c = U2 @ has
    out['np_2_mean'] = np.where(c > 0, (U2 @ pv) / np.maximum(c, 1), -1)
    return pd.DataFrame({k: v[ix] for k, v in out.items()})


def tree_model(a, b, feat_cols, seeds=(0,), stack=True, rec_halflife=8, params=BASE_LGB, nfold=5,
               extra_a=None, extra_b=None, nrounds=1):
    cut = a.time_step.max()
    w = (0.5 ** ((cut - a.time_step) / rec_halflife)).values if rec_halflife else np.ones(len(a))
    outp = []
    for sd in seeds:
        pb = lgbm(a[feat_cols], a.label, w, sd, params).predict_proba(b[feat_cols])[:, 1]
        if stack:
            Xa, Xb = a[feat_cols].reset_index(drop=True), b[feat_cols].reset_index(drop=True)
            for rnd in range(nrounds):
                oof = np.zeros(len(a))
                for fold in np.array_split(np.sort(a.time_step.unique()), nfold):
                    mk = a.time_step.isin(fold).values
                    oof[mk] = lgbm(Xa.loc[~mk], a.label[~mk], w[~mk], sd, params).predict_proba(Xa.loc[mk])[:, 1]
                p = np.full(N, np.nan); p[a.ix] = oof; p[b.ix] = pb
                suffix = '' if rnd == 0 else f'_r{rnd+1}'
                Xa = pd.concat([Xa, nb_feats(p, a.ix.values).add_suffix(suffix)], axis=1)
                Xb = pd.concat([Xb, nb_feats(p, b.ix.values).add_suffix(suffix)], axis=1)
                if rnd == 0 and extra_a is not None:
                    Xa = Xa.copy(); Xa['gnn_mean'] = extra_a
                    Xb = Xb.copy(); Xb['gnn_mean'] = extra_b
                pb = lgbm(Xa, a.label, w, sd, params).predict_proba(Xb)[:, 1]
        outp.append(pb)
    return np.mean(outp, 0)


def rk(p): return rankdata(p) / len(p)


## 6. Model 1 vs Model 2: does stacking help, on the full feature set?

In [9]:
print('Model 1 (LGBM only) vs Model 2 (LGBM + stacking + neighbor preds):')
for nm, (cut, hor) in WINDOWS.items():
    a, b = split(cut, hor)
    p1 = tree_model(a, b, XF_ALL, stack=False)
    p2 = tree_model(a, b, XF_ALL, stack=True, nrounds=1)
    print(f'  {nm:16s} m1(no-stack): PR={ap(b.label,p1):.4f} ROC={auc(b.label,p1):.4f}   '
          f'm2(+stack): PR={ap(b.label,p2):.4f} ROC={auc(b.label,p2):.4f}')


Model 1 (LGBM only) vs Model 2 (LGBM + stacking + neighbor preds):
  holdout_7@28     m1(no-stack): PR=0.9904 ROC=0.9974   m2(+stack): PR=0.9945 ROC=0.9984
  stress_7@22      m1(no-stack): PR=0.7416 ROC=0.8390   m2(+stack): PR=0.7463 ROC=0.8490
  horizon14@14     m1(no-stack): PR=0.9302 ROC=0.9630   m2(+stack): PR=0.9367 ROC=0.9731
  horizon14@17     m1(no-stack): PR=0.8261 ROC=0.9008   m2(+stack): PR=0.8352 ROC=0.9158
  horizon14@21     m1(no-stack): PR=0.8122 ROC=0.8971   m2(+stack): PR=0.8161 ROC=0.9011


Observation: Stacking helps as it wins on all windows

## 7. Feature selection on old+new features

In [10]:
a, b = split(*WINDOWS['horizon14@21'])
cut = a.time_step.max(); w = (0.5 ** ((cut - a.time_step) / 8)).values
imp_model = lgbm(a[XF_ALL], a.label, w, 0)
importance = pd.Series(imp_model.feature_importances_, index=XF_ALL).sort_values()

best_k, best_score = None, -1
for k in [40, 60, 80, 100, 120, 140]:
    bottomk = importance.head(k).index.tolist()
    keep = [c for c in XF_ALL if c not in set(bottomk)]
    prs, rocs = [], []
    for nm in HARD_WINDOWS:
        a2, b2 = split(*WINDOWS[nm])
        pred = tree_model(a2, b2, keep, stack=True, nrounds=1)
        prs.append(ap(b2.label, pred)); rocs.append(auc(b2.label, pred))
    mean_pr, mean_roc = np.mean(prs), np.mean(rocs)
    combined = (mean_pr + mean_roc) / 2
    print(f'  drop bottom-{k} (n={len(keep):3d})  mean-PR={mean_pr:.4f} mean-ROC={mean_roc:.4f} '
          f'combined={combined:.4f}  per-window-ROC={[round(r,4) for r in rocs]}  per-window-PR={[round(r,4) for r in prs]}')
    if combined > best_score:
        best_score, best_k = combined, k

bottomK = importance.head(best_k).index.tolist()
XF_FINAL = [c for c in XF_ALL if c not in set(bottomK)]
print(f'\nselected cutoff K={best_k} -> XF_FINAL: {len(XF_FINAL)} features (from {len(XF_ALL)})')


  drop bottom-40 (n=132)  mean-PR=0.8346 mean-ROC=0.9108 combined=0.8727  per-window-ROC=[0.86, 0.9709, 0.9251, 0.8871]  per-window-PR=[0.7519, 0.9342, 0.8415, 0.8108]
  drop bottom-60 (n=112)  mean-PR=0.8376 mean-ROC=0.9179 combined=0.8778  per-window-ROC=[0.8769, 0.9658, 0.9218, 0.9071]  per-window-PR=[0.7608, 0.9318, 0.8392, 0.8188]
  drop bottom-80 (n= 92)  mean-PR=0.8369 mean-ROC=0.9159 combined=0.8764  per-window-ROC=[0.8654, 0.9697, 0.925, 0.9035]  per-window-PR=[0.7552, 0.9343, 0.8409, 0.8173]
  drop bottom-100 (n= 72)  mean-PR=0.8337 mean-ROC=0.9102 combined=0.8720  per-window-ROC=[0.8501, 0.9659, 0.9166, 0.9081]  per-window-PR=[0.7476, 0.9316, 0.8362, 0.8196]
  drop bottom-120 (n= 52)  mean-PR=0.8285 mean-ROC=0.9051 combined=0.8668  per-window-ROC=[0.8747, 0.966, 0.8905, 0.8892]  per-window-PR=[0.7583, 0.927, 0.8209, 0.808]
  drop bottom-140 (n= 32)  mean-PR=0.8241 mean-ROC=0.9099 combined=0.8670  per-window-ROC=[0.8676, 0.9673, 0.9015, 0.9033]  per-window-PR=[0.7484, 0.924, 

k=60 selected as combined is higher

### CHeck: all features vs dropped

In [11]:
print('Robustness check across ALL windows (all-features vs XF_FINAL):')
for nm, (cut_, hor_) in WINDOWS.items():
    a2, b2 = split(cut_, hor_)
    p_base = tree_model(a2, b2, XF_ALL, stack=True, nrounds=1)
    p_final = tree_model(a2, b2, XF_FINAL, stack=True, nrounds=1)
    print(f'  {nm:16s} all-features: PR={ap(b2.label,p_base):.4f} ROC={auc(b2.label,p_base):.4f}   '
          f'XF_FINAL: PR={ap(b2.label,p_final):.4f} ROC={auc(b2.label,p_final):.4f}')


Robustness check across ALL windows (all-features vs XF_FINAL):
  holdout_7@28     all-features: PR=0.9945 ROC=0.9984   XF_FINAL: PR=0.9950 ROC=0.9985
  stress_7@22      all-features: PR=0.7463 ROC=0.8490   XF_FINAL: PR=0.7608 ROC=0.8769
  horizon14@14     all-features: PR=0.9367 ROC=0.9731   XF_FINAL: PR=0.9318 ROC=0.9658
  horizon14@17     all-features: PR=0.8352 ROC=0.9158   XF_FINAL: PR=0.8392 ROC=0.9218
  horizon14@21     all-features: PR=0.8161 ROC=0.9011   XF_FINAL: PR=0.8188 ROC=0.9071


## 8. LGBM regularization tuning

A small grid over L1/L2 regularization and minimum leaf size, on `XF_FINAL`, checked across all
four hard windows -- a config only replaces the baseline if it wins or ties on **both** PR-AUC and
ROC-AUC, everywhere.

In [12]:
LGB_GRID = {
    'baseline':        dict(reg_alpha=0.0, reg_lambda=0.0, min_child_samples=20),
    'l2_reg':          dict(reg_alpha=0.0, reg_lambda=1.0, min_child_samples=20),
    'l1_l2_reg':        dict(reg_alpha=0.1, reg_lambda=0.1, min_child_samples=20),
    'bigger_min_leaf': dict(reg_alpha=0.0, reg_lambda=0.0, min_child_samples=50),
}
lgb_results = {}
for name, extra in LGB_GRID.items():
    params = {**BASE_LGB, **extra}
    metrics = {}
    for nm in HARD_WINDOWS:
        a2, b2 = split(*WINDOWS[nm])
        pred = tree_model(a2, b2, XF_FINAL, stack=True, nrounds=1, params=params)
        metrics[nm] = (ap(b2.label, pred), auc(b2.label, pred))
    lgb_results[name] = metrics
    print(f'  {name:16s} ' + '  '.join(f'{k}=(PR={v[0]:.4f},ROC={v[1]:.4f})' for k, v in metrics.items()))

base_metrics = lgb_results['baseline']
BEST_LGB_NAME = 'baseline'
for name in LGB_GRID:
    if name == 'baseline':
        continue
    if all(lgb_results[name][w][0] >= base_metrics[w][0] and lgb_results[name][w][1] >= base_metrics[w][1]
           for w in HARD_WINDOWS):
        BEST_LGB_NAME = name
        base_metrics = lgb_results[name]
BASE_LGB_FINAL = {**BASE_LGB, **LGB_GRID[BEST_LGB_NAME]}
print(f'-> selected LGBM config: {BEST_LGB_NAME} = {LGB_GRID[BEST_LGB_NAME]}')


  baseline         stress_7@22=(PR=0.7608,ROC=0.8769)  horizon14@14=(PR=0.9318,ROC=0.9658)  horizon14@17=(PR=0.8392,ROC=0.9218)  horizon14@21=(PR=0.8188,ROC=0.9071)
  l2_reg           stress_7@22=(PR=0.7342,ROC=0.8145)  horizon14@14=(PR=0.9312,ROC=0.9622)  horizon14@17=(PR=0.8351,ROC=0.9142)  horizon14@21=(PR=0.7984,ROC=0.8544)
  l1_l2_reg        stress_7@22=(PR=0.7299,ROC=0.8026)  horizon14@14=(PR=0.9295,ROC=0.9583)  horizon14@17=(PR=0.8284,ROC=0.9008)  horizon14@21=(PR=0.8036,ROC=0.8726)
  bigger_min_leaf  stress_7@22=(PR=0.7511,ROC=0.8578)  horizon14@14=(PR=0.9327,ROC=0.9691)  horizon14@17=(PR=0.8347,ROC=0.9141)  horizon14@21=(PR=0.8204,ROC=0.9088)
-> selected LGBM config: baseline = {'reg_alpha': 0.0, 'reg_lambda': 0.0, 'min_child_samples': 20}


{'reg_alpha': 0.0, 'reg_lambda': 0.0, 'min_child_samples': 20}

## 9. Does a second stacking round help, with the tuned config + XF_FINAL?

Feature selection changes what's left for a second round to build on, so this is rechecked here
rather than assumed from the earlier notebook's conclusion.

In [ ]:
print('nrounds=1 vs nrounds=2 (XF_FINAL, tuned LGBM):')
n2_wins = 0
for nm, (cut_, hor_) in WINDOWS.items():
    a2, b2 = split(cut_, hor_)
    p1 = tree_model(a2, b2, XF_FINAL, stack=True, nrounds=1, params={'reg_alpha': 0.0, 'reg_lambda': 0.0, 'min_child_samples': 20})
    p2 = tree_model(a2, b2, XF_FINAL, stack=True, nrounds=2, params={'reg_alpha': 0.0, 'reg_lambda': 0.0, 'min_child_samples': 20})
    pr1, r1 = ap(b2.label, p1), auc(b2.label, p1)
    pr2, r2 = ap(b2.label, p2), auc(b2.label, p2)
    print(f'  {nm:16s} n1: PR={pr1:.4f} ROC={r1:.4f}   n2: PR={pr2:.4f} ROC={r2:.4f}')
    if pr2 >= pr1 and r2 >= r1:
        n2_wins += 1
NROUNDS_FINAL = 2 if n2_wins == len(WINDOWS) else 1
print(f'-> using nrounds={NROUNDS_FINAL}')


nrounds=1 vs nrounds=2 (XF_FINAL, tuned LGBM):


  File "c:\Users\ongch\anaconda3\lib\site-packages\joblib\externals\loky\backend\context.py", line 227, in _count_physical_cores
    cpu_info = subprocess.run(
  File "c:\Users\ongch\anaconda3\lib\subprocess.py", line 505, in run
    with Popen(*popenargs, **kwargs) as process:
  File "c:\Users\ongch\anaconda3\lib\subprocess.py", line 951, in __init__
    self._execute_child(args, executable, preexec_fn, close_fds,
  File "c:\Users\ongch\anaconda3\lib\subprocess.py", line 1420, in _execute_child
    hp, ht, pid, tid = _winapi.CreateProcess(executable, args,


  holdout_7@28     n1: PR=0.9930 ROC=0.9977   n2: PR=0.9929 ROC=0.9978
  stress_7@22      n1: PR=0.7548 ROC=0.8753   n2: PR=0.7397 ROC=0.8395
  horizon14@14     n1: PR=0.9215 ROC=0.9663   n2: PR=0.9205 ROC=0.9670
  horizon14@17     n1: PR=0.8316 ROC=0.9172   n2: PR=0.8346 ROC=0.9224
  horizon14@21     n1: PR=0.8211 ROC=0.9194   n2: PR=0.8175 ROC=0.9122
-> using nrounds=1 (n2 won-or-tied on BOTH metrics on 1/5 windows; all required)


nrounds = 1 used as it wins on most windows

## 10. GNN setup

In [14]:
def node_matrix(mask_after=None, mask_all_unknown=True):
    Xn = pd.DataFrame({'txId': ids}).merge(pd.concat([train[['txId', 'label'] + F_OLD], test[['txId'] + F_OLD]]), on='txId', how='left')
    tstep = pd.DataFrame({'txId': ids}).merge(pd.concat([train[['txId', 'time_step']], test[['txId', 'time_step']]]), on='txId', how='left').time_step.values
    X = Xn[F_OLD].values.astype(np.float32); X = np.sign(X) * np.log1p(np.abs(X))
    hf = ~np.isnan(X[:, 0]); unk_train = Xn.label.isna().values & np.isin(ids, train.txId.values)
    if mask_after is not None: hf &= ~(unk_train & (tstep > mask_after))
    if mask_all_unknown: hf &= ~unk_train
    X[~hf] = 0; X = np.nan_to_num(X)
    S = np.log1p(G[STRUCT_NEW].clip(lower=0).values.astype(np.float32))
    return torch.tensor(np.hstack([X, S, hf[:, None].astype(np.float32)]))


EI = to_undirected(torch.tensor(np.vstack([src, dst]), dtype=torch.long))


class GNN(nn.Module):
    def __init__(s, kind, din, h=64, heads=4, drop=0.3):
        super().__init__(); s.drop = drop; s.inp = nn.Linear(din, h)
        conv = {'gcn': lambda: GCNConv(h, h), 'sage': lambda: SAGEConv(h, h),
                'gat': lambda: GATConv(h, h // heads, heads=heads)}[kind]
        s.c1, s.c2 = conv(), conv()
        s.out = nn.Sequential(nn.Linear(3 * h, h), nn.ReLU(), nn.Dropout(drop), nn.Linear(h, 1))

    def forward(s, x, ei):
        h0 = Fn.relu(s.inp(x))
        h1 = Fn.relu(s.c1(Fn.dropout(h0, s.drop, s.training), ei))
        h2 = Fn.relu(s.c2(Fn.dropout(h1, s.drop, s.training), ei))
        return s.out(torch.cat([h0, h1, h2], 1)).squeeze(-1)


def run_gnn(kind, X, train_ix, score_ix, y_train, epochs=150, seed=0, lr=0.005, h=64, drop=0.3, wd=5e-4):
    seeds_ = torch.tensor(np.unique(np.concatenate([train_ix, score_ix])))
    sub, ei, mapping, _ = k_hop_subgraph(seeds_, 2, EI, relabel_nodes=True, num_nodes=N)
    loc = pd.Series(np.arange(len(sub)), index=sub.numpy())
    tri, sci = loc[train_ix].values, loc[score_ix].values; xs = X[sub]
    torch.manual_seed(seed); m = GNN(kind, X.shape[1], h=h, drop=drop); opt = torch.optim.AdamW(m.parameters(), lr=lr, weight_decay=wd)
    yt = torch.tensor(y_train, dtype=torch.float32)
    for ep in range(epochs):
        m.train(); opt.zero_grad()
        loss = Fn.binary_cross_entropy_with_logits(m(xs, ei)[tri], yt)
        loss.backward(); opt.step()
    m.eval()
    with torch.no_grad():
        return torch.sigmoid(m(xs, ei)[sci]).numpy()


def gnn_mean_for(cut, hor, gnn_kwargs):
    a, b = split(cut, hor)
    X_window = node_matrix(mask_after=cut)
    gnn_b = {kind: run_gnn(kind, X_window, a.ix.values, b.ix.values, a.label.values, **gnn_kwargs) for kind in ['gcn', 'sage', 'gat']}
    gnn_mean_b = (rk(gnn_b['gcn']) + rk(gnn_b['sage']) + rk(gnn_b['gat'])) / 3
    gnn_a = {kind: run_gnn(kind, X_window, a.ix.values, a.ix.values, a.label.values, **gnn_kwargs) for kind in ['gcn', 'sage', 'gat']}
    gnn_mean_a = (rk(gnn_a['gcn']) + rk(gnn_a['sage']) + rk(gnn_a['gat'])) / 3
    return gnn_mean_a, gnn_mean_b


## 11. GNN hyperparameter tuning (dropout, weight decay)

Checked on the two most informative hard windows, to keep this tractable -- GNN training is the
most compute-heavy part of this notebook.

In [ ]:
GNN_GRID = {
    'baseline':      dict(h=64, drop=0.3, wd=5e-4, lr=0.005, epochs=150),
    'lower_dropout': dict(h=64, drop=0.1, wd=5e-4, lr=0.005, epochs=150),
    'higher_wd':     dict(h=64, drop=0.1, wd=1e-3, lr=0.005, epochs=150),
}
GNN_TUNE_WINDOWS = ['stress_7@22', 'horizon14@21']
gnn_tune_results = {}
for name, kwargs in GNN_GRID.items():
    metrics = {}
    for nm in GNN_TUNE_WINDOWS:
        cut, hor = WINDOWS[nm]
        _, gnn_mean_b = gnn_mean_for(cut, hor, kwargs)
        _, b = split(cut, hor)
        metrics[nm] = (ap(b.label, gnn_mean_b), auc(b.label, gnn_mean_b))
    gnn_tune_results[name] = metrics
    print(f'  {name:16s} ' + '  '.join(f'{k}=(PR={v[0]:.4f},ROC={v[1]:.4f})' for k, v in metrics.items()))

base_g = gnn_tune_results['baseline']
BEST_GNN_NAME = 'baseline'
for name in GNN_GRID:
    if name == 'baseline':
        continue
    if all(gnn_tune_results[name][w][0] >= base_g[w][0] and gnn_tune_results[name][w][1] >= base_g[w][1]
           for w in GNN_TUNE_WINDOWS):
        BEST_GNN_NAME = name
        base_g = gnn_tune_results[name]
GNN_KWARGS_FINAL = GNN_GRID[BEST_GNN_NAME]
print(f'-> selected GNN config: {BEST_GNN_NAME} = {GNN_KWARGS_FINAL}')


selected GNN config: lower_dropout = {'h': 64, 'drop': 0.1, 'wd': 0.0005, 'lr': 0.005, 'epochs': 150}

## 12. Four-way model comparison (tuned LGBM + tuned GNN, XF_FINAL, all hard windows)

The decisive comparison: m1 (LGBM only), m2 (LGBM + stacking), m3 (m2 + fixed-weight GNN blend),
m4 (m2 + GNN joint-stack). A GNN-based variant only replaces m2 as the final model if it wins or
ties on **both PR-AUC and ROC-AUC**, on **every** hard window -- not just the easy one, and not
just one of the two competition-relevant metrics.

In [ ]:
model_results = {}
for nm in HARD_WINDOWS + ['holdout_7@28']:
    cut, hor = WINDOWS[nm]
    a, b = split(cut, hor)
    gnn_mean_a, gnn_mean_b = gnn_mean_for(cut, hor, {'h': 64, 'drop': 0.1, 'wd': 0.0005, 'lr': 0.005, 'epochs': 150})

    p1 = tree_model(a, b, XF_FINAL, stack=False, params=BASE_LGB_FINAL)
    p2 = tree_model(a, b, XF_FINAL, stack=True, nrounds=1, params=BASE_LGB_FINAL)
    p3 = 0.8 * rk(p2) + 0.2 * gnn_mean_b
    p4 = tree_model(a, b, XF_FINAL, stack=True, nrounds=1, params=BASE_LGB_FINAL,
                     extra_a=gnn_mean_a, extra_b=gnn_mean_b)

    model_results[nm] = {
        'm1_lgbm_only':        (ap(b.label, p1), auc(b.label, p1)),
        'm2_lgbm_stack':       (ap(b.label, p2), auc(b.label, p2)),
        'm3_stack_gnn_blend':  (ap(b.label, p3), auc(b.label, p3)),
        'm4_stack_gnn_joint':  (ap(b.label, p4), auc(b.label, p4)),
    }
    print(f'\n{nm}:')
    for k, (pr, rc) in model_results[nm].items():
        print(f'  {k:20s} PR={pr:.4f} ROC={rc:.4f}')


Final 2 Model is M2_LGBM_stack and m3_stack_gnn_blend as it wins on most validation horizons

## 13. Final fit on all labeled data

In [ ]:
# Shared tree part
tree_preds = []
for sd in range(5):
    pred = tree_model(L, T, XF_FINAL, seeds=(sd,), stack=True,
                      nrounds=NROUNDS_FINAL, params=BASE_LGB_FINAL)
    tree_preds.append(pred)
final_pred_m2 = np.mean(tree_preds, 0)

# GNN part (for m3)
X_full = node_matrix()
gnn_preds = {}
for kind in ['gcn', 'sage', 'gat']:
    seed_preds = []
    for sd in (0, 1):
        seed_preds.append(run_gnn(kind, X_full, L.ix.values, T.ix.values, L.label.values,
                                  seed=sd, **{'h': 64, 'drop': 0.1, 'wd': 0.0005, 'lr': 0.005, 'epochs': 150}))
    gnn_preds[kind] = np.mean(seed_preds, 0)
    print(f'{kind} done')
gnn_mean_test = (rk(gnn_preds['gcn']) + rk(gnn_preds['sage']) + rk(gnn_preds['gat'])) / 3

# m3 blend
final_pred_m3 = 0.8 * rk(final_pred_m2) + 0.2 * gnn_mean_test

# Report + write both submissions
for name, pred in [('m2_lgbm_stack', final_pred_m2), ('m3_stack_gnn_blend', final_pred_m3)]:
    print(f'{name}: n={len(pred)} mean={pred.mean():.4f} min={pred.min():.4f} max={pred.max():.4f}')
    sub = pd.DataFrame({'index': T['index'].values, 'target': pred}).sort_values('index').reset_index(drop=True)
    assert len(sub) == len(sample) and (sub['index'].values == sample['index'].values).all()
    assert sub.target.between(0, 1).all()
    sub.to_csv(f'submission_{name}.csv', index=False)
    print(f'  wrote submission_{name}.csv')